In [1]:
!pip install mne -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 95.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [2]:
import mne
import numpy as np

In [3]:
from mne.datasets import eegbci

In [4]:
# Connect to Google drive

from google.colab import drive
import os

drive.mount("/content/drive", force_remount = True)


PROJECT_DIR = "/content/drive/MyDrive/neurosynth"

os.makedirs(PROJECT_DIR, exist_ok = True)

Mounted at /content/drive


In [5]:
# Loading the dataset

DATA_DIR = "/content/drive/MyDrive/neurosynth/data/raw"
OUTPUT_DIR = "/content/drive/MyDrive/neurosynth/data/processed"



SUBJECTS = [1, 2, 3, 4, 5]
RUNS = [4, 8, 12]


# Bandpass filter - keeps only Alpha + Beta motor imagery frequencies
# validated earlier with our PSD plot
FREQ_LOW = 8.0
FREQ_HIGH = 30.0

# Epoch window - 4 second task windows matching experiment design
EPOCH_TMIN = 0.0
EPOCH_TMAX = 4.0

# T1 - left fist imagery --> label 0
# T2 - right fist imagery ----> label 1
EVENT_ID = {"T1": 1, "T2": 2}


os.makedirs(OUTPUT_DIR, exist_ok = True)
print("Configuration set!")


Configuration set!


In [6]:
# see exactly what files exist for one subject
from mne.datasets import eegbci


# request ALL 14 runs to see the full picture
all_runs = list(range(1,15)) # 1 to 14

all_fnames = eegbci.load_data(
    1,
    runs = all_runs,
    path = DATA_DIR,
    verbose = False
)


print("All 14 runs for Subject 1")
for i, fname in enumerate(all_fnames, start = 1):
  print(f"Run {i:2d}: {fname}")

Do you want to set the path:
    /content/drive/MyDrive/neurosynth/data/raw
as the default EEGBCI dataset path in the mne-python config [y]/n? y
All 14 runs for Subject 1
Run  1: /content/drive/MyDrive/neurosynth/data/raw/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R01.edf
Run  2: /content/drive/MyDrive/neurosynth/data/raw/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R02.edf
Run  3: /content/drive/MyDrive/neurosynth/data/raw/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R03.edf
Run  4: /content/drive/MyDrive/neurosynth/data/raw/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf
Run  5: /content/drive/MyDrive/neurosynth/data/raw/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R05.edf
Run  6: /content/drive/MyDrive/neurosynth/data/raw/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R06.edf
Run  7: /content/drive/MyDrive/neurosynth/data/raw/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R07.edf
Run  8: /content/drive/MyDrive/neurosynth/data/raw/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S

In [7]:
# load run 4 specifically and check its annotations
raw_check = mne.io.read_raw_edf(
    "/content/drive/MyDrive/neurosynth/data/raw/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf",
    preload = True, verbose = False

)



print("Annotations in Run 4:")
for ann in raw_check.annotations[:6]:
  print(f"    {ann['onset']:.1f}s -->'{ann['description']}'")

Annotations in Run 4:
    0.0s -->'T0'
    4.2s -->'T2'
    8.3s -->'T0'
    12.5s -->'T1'
    16.6s -->'T0'
    20.8s -->'T1'


In [8]:
# Preprocessing Functions

from mne.io import concatenate_raws
from tqdm.notebook import tqdm
def load_subject(subject, runs, data_dir):
  """
  Loads all runs for one subject and joins them into
  one continuous EEG recording
  """

  raw_fnames = eegbci.load_data(
      subjects = subject,
      runs = runs,
      path = data_dir,
      verbose = False
  )

  raws = [
      mne.io.read_raw_edf(
          f,
          preload = True,
          verbose = False

      )
      for f in raw_fnames
  ]


  raw = concatenate_raws(raws)



  # set channel names to standard format (removes trailling dots)
  eegbci.standardize(raw)


  # set electrode scalp positions - needed for spartial operations
  montage = mne.channels.make_standard_montage("standard_1005")
  raw.set_montage(montage, verbose = False)
  return raw



def apply_common_average_reference(raw):

  """
  Applies Common Average Reference (CAR)

  subtract the average signal across ALL channels from each
  individual channel at every timepoint This removes noise/
  artifacts shared across the whole scalp

  """

  raw_car = raw.copy()
  raw_car.set_eeg_reference("average", verbose = False)
  return raw_car



def apply_bandpass_filter(raw):
  """
  Keeps only 8- 30Hz - Alpha + Beta motor imagery bands.
  Validated earlier with our PSD plot showing real actuvity
  in this range plut a power-line spike outside it

  """

  raw.filter (
      l_freq = FREQ_LOW,
      h_freq = FREQ_HIGH,
      method = 'iir',
      verbose = False

  )

  return raw


def extract_epoch(raw):
  """
   Cuts the continuous signal into 4-secon windows
aligned to each T1/T2 event onset.
    """
  events, _ = mne.events_from_annotations(raw,
                                          event_id = EVENT_ID,
                                          verbose = False)
  epochs = mne.Epochs(
       raw,
       events,
       event_id = EVENT_ID,
       tmin = EPOCH_TMIN,
       tmax = EPOCH_TMAX,
       baseline = None,
       preload = True,
       verbose = False

   )

  X = epochs.get_data()
  y = epochs.events[:, 2] -1
  return X, y


def normalize(X):
    """
     Z - score normalization per channel per epoch.
     Removes aplitude difference betwen subject so the
     model learns signal patterns, not raw voltage scale

    """
    mean = X.mean(axis = 1, keepdims = True)
    std = X.std(axis = 1, keepdims = True)
    std[std ==0] = 1
    return (X - mean) /std



print("All preprocessing functions defined!")

All preprocessing functions defined!


In [9]:
help(eegbci.load_data)

Help on function load_data in module mne.datasets.eegbci.eegbci:

load_data(subjects, runs, *, path=None, force_update=False, update_path=None, base_url='https://physionet.org/files/eegmmidb/1.0.0/', verbose=None)
    Get paths to local copies of EEGBCI dataset files.

    This will fetch data for the EEGBCI dataset :footcite:`SchalkEtAl2004`, which is
    also available at PhysioNet :footcite:`GoldbergerEtAl2000`. Metadata, such as the
    meaning of event markers may be obtained from the
    `PhysioNet documentation page <https://physionet.org/content/eegmmidb/1.0.0/>`_.

    Parameters
    ----------
    subjects : int | list of int
        The subjects to use. Can be in the range of 1-109 (inclusive).
    runs : int | list of int
        The runs to use (see Notes for details).
    path : None | path-like
        Location of where to look for the EEGBCI data. If ``None``, the environment
        variable or config parameter ``MNE_DATASETS_EEGBCI_PATH`` is used. If neither
        e

In [ ]:
all_X = []
all_y  = []



print("Starting preprocessing pipeline...")

for subject in tqdm(SUBJECTS, desc = "Processing subjects"):



   # 1. Load the data
   raw = load_subject(subject, RUNS, DATA_DIR)

   # 2. Applying Common Average Reference
   raw = apply_common_average_reference(raw)

   # 3. Applying the bandpass filter
   raw = apply_bandpass_filter(raw)


   # 4. Epoch (4 - seconds window)

   X, y = extract_epoch(raw)


   if len(X) <5 :
    print(f" Subject {subject} skipped -- too few epochs")
    continue

   # 5. Normalize
   X = normalize(X)

   all_X.append(X)
   all_y.append(y)

   print(f" Subject {subject:03d} - {X.shape[0]} epochs|. shape {X.shape}")



all_X = np.concatenate(all_X, axis = 0)
all_y = np.concatenate(all_y, axis = 0)


print("Pipeline Complete!")
print(f"  Total epochs: {all_X.shape[0]}")
print(f"  X shape: {all_X.shape}")
print(f"  y shape: {all_y.shape}")
print(f"  Left fist:  {(all_y ==0).sum()} epochs")
print(f"  Right fist: {(all_y ==1).sum()} epochs")

Starting preprocessing pipeline...


Processing subjects:   0%|          | 0/5 [00:00<?, ?it/s]

 Subject 001 - 45 epochs|. shape (45, 64, 641)
 Subject 002 - 45 epochs|. shape (45, 64, 641)
 Subject 003 - 45 epochs|. shape (45, 64, 641)
 Subject 004 - 45 epochs|. shape (45, 64, 641)


In [ ]:
# Check one subject's raw event distribution
raw_check = load_subject(1, RUNS, DATA_DIR)
events_check, _ = mne.events_from_annotations(raw_check, verbose = False)

import numpy as np
unique, counts = np.unique(events_check[:, 2], return_counts = True)
print("Raw event ID counts:", dict(zip(unique, counts)))

Raw event ID counts: {np.int64(1): np.int64(45), np.int64(2): np.int64(23), np.int64(3): np.int64(22)}


In [ ]:
# Save corrected data to drive

OUTPUT_DIR = "/content/drive/MyDrive/neurosynth/data/processed"

np.save(f"{OUTPUT_DIR}/X.npy", all_X)
np.save(f"{OUTPUT_DIR}/y.npy", all_y)


print("Corrected data saved to Google Drive!")
print(f"   {OUTPUT_DIR}/X.npy")
print(f"   {OUTPUT_DIR}/y.npy")

Corrected data saved to Google Drive!
   /content/drive/MyDrive/neurosynth/data/processed/X.npy
   /content/drive/MyDrive/neurosynth/data/processed/y.npy


In [ ]:
# Verifying the saved files

X_check = np.load(f"{OUTPUT_DIR}/X.npy")
y_check = np.load(f"{OUTPUT_DIR}/y.npy")

print("X shape:", X_check.shape)
print("y shape:", y_check.shape)
print(f"Total epochs: {X_check.shape[0]}")
print(f"Left fist:  {(y_check ==0).sum()} epochs")
print(f"Right fist: {(y_check ==1).sum()} epochs")
print(f"Data mean:  {X_check.mean():.6f}")
print(f"Data std:   {X_check.std():.6f}")



if X_check.shape[0] == 225:
    print("\n Confirmed — this is the corrected data (225 epochs)")
else:
    print(f"\n Unexpected count: {X_check.shape[0]} — something's off")

X shape: (225, 64, 641)
y shape: (225,)
Total epochs: 225
Left fist:  113 epochs
Right fist: 112 epochs
Data mean:  0.000000
Data std:   1.000000

 Confirmed — this is the corrected data (225 epochs)


In [ ]:
import json
# shutil = copy files between folders
import shutil
import os
import subprocess

# datetime = get current date/time for commit messages
from datetime import datetime
from google.colab import userdata


GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
GITHUB_USERNAME = "the-liyanage"
REPO_NAME = "neurosynth"


# 1. set git identity
subprocess.run(["git", "config", "--global",
               "user.name", "the-liyanage"])

subprocess.run(["git", "config", "--global",
                "user.email", "hiruniliyanage4@gmail.com"])


# 2. Clone repo fresh (only if not already there)
if not os.path.exists(f"/content/{REPO_NAME}/.git"):
  subprocess.run(["rm", "-rf", "f/content/{REPO_NAME}"])
  os.chdir("/content")
  subprocess.run([
      "git", "clone",
      f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
      ])
  print("Repo clones fresh!")
else:
  print("Repo already exisit, skipping clone")




# 3. Copy notebook into repo
os.makedirs(f"/content/{REPO_NAME}/notebooks", exist_ok = True)
shutil.copy(
    "/content/drive/MyDrive/Colab Notebooks/02_preprocessing.ipynb",
    f"/content/neurosynth/notebooks/02_preprocessing.ipynb"
)
print(f"\n Notebook copied!")


# 4. Commit and push

# move into the repo folder
os.chdir(f"/content/{REPO_NAME}")

# stage all change
subprocess.run(["git", "pull"])
subprocess.run(["git", "add", "."])


commit_message = " done preprocessing "

subprocess.run(["git", "commit", "-m", commit_message])


result = subprocess.run([
    "git", "push",
    f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git",
    "main"
], capture_output=True, text=True)


if result.returncode == 0:
  print("Pushed to the github", commit_message)
else:
  print("Push failed")
  print(result.stderr)

Repo already exisit, skipping clone

 Notebook copied!
Pushed to the github  done preprocessing 
